In [10]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [11]:
# Pin to a single GPU BEFORE importing torch/transformers/anything CUDA-related.
# Kaggle gives you 2x T4 by default; without this, HF Trainer auto-wraps the model in
# nn.DataParallel because it sees n_gpu > 1, which then crashes because the quantized
# model weights only live on cuda:0.
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"


In [12]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))


/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [13]:
!pip install -Uq "trl[peft]" trackio bitsandbytes liger-kernel


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 61.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 32.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 403.1/403.1 kB 27.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 87.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 838.8/838.8 kB 48.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible

In [14]:
import pandas as pd
from datasets import Dataset

train = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
test  = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")
sample_submission = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv")

print(train.shape, test.shape, sample_submission.shape)
train.head()


(2000, 8) (500, 7) (500, 2)


,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C
3,4,Select the most accurate option: What is Marti...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
4,5,Identify the correct statement: What is the co...,"Simultaneity is relative, meaning that two eve...","Simultaneity is relative, meaning that two eve...","Simultaneity is absolute, meaning that two eve...",Simultaneity is a concept that applies only to...,Simultaneity is a concept that applies only to...,A


In [15]:
OPTION_LETTERS = ["A", "B", "C", "D", "E"]

def build_prompt(row):
    return f"""You are an expert at solving multiple-choice questions.

Choose the single best answer from the five options.

Question:
{row['prompt']}

Options:
A. {row['A']}
B. {row['B']}
C. {row['C']}
D. {row['D']}
E. {row['E']}

Respond with exactly one uppercase letter: A, B, C, D, or E.

Answer:"""

def build_messages(row):
    return [
        {"role": "user", "content": build_prompt(row)},
        {"role": "assistant", "content": row["answer"]},
    ]


In [16]:
rows = []

for _, r in train.iterrows():
    rows.append({"messages": build_messages(r)})

dataset = Dataset.from_list(rows)

dataset = dataset.train_test_split(test_size=0.1, seed=42)

train_dataset = dataset["train"]
eval_dataset = dataset["test"]

print(train_dataset)
print(eval_dataset)


Dataset({
    features: ['messages'],
    num_rows: 1800
})
Dataset({
    features: ['messages'],
    num_rows: 200
})


In [17]:
# Select one model below by uncommenting the line you want to use 👇
## Qwen
model_id = "Qwen/Qwen2.5-3B-Instruct"

output_dir = "Qwen2.5-MCQ"     # ⚠️ ~14.1 GB VRAM
# model_id, output_dir = "Qwen/Qwen3-8B", "Qwen3-8B-SFT"                                          # ⚠️ ~12.8 GB VRAM
# model_id, output_dir = "Qwen/Qwen2.5-7B-Instruct", "Qwen2.5-7B-Instruct"                        # ✅ ~10.8 GB VRAM

## Llama
# model_id, output_dir = "meta-llama/Llama-3.2-3B-Instruct", "Llama-3.2-3B-Instruct"              # ✅ ~4.7 GB VRAM
# model_id, output_dir = "meta-llama/Llama-3.1-8B-Instruct", "Llama-3.1-8B-Instruct"              # ⚠️ ~10.9 GB VRAM

## Gemma
# model_id, output_dir = "google/gemma-3n-E2B-it", "gemma-3n-E2B-it"                              # ❌ Upgrade to a higher tier of colab
# model_id, output_dir = "google/gemma-3-4b-it", "gemma-3-4b-it"                                  # ⚠️ ~6.8 GB VRAM

## Granite
#model_id, output_dir = "ibm-granite/granite-4.0-micro", "granite-4.0-micro"                      # ✅ ~3.3 GB VRAM

## LFM2
#model_id, output_dir = "LiquidAI/LFM2-2.6B", "LFM2-2.6B-SFT"                                     # ✅ ~5.89 GB VRAM


In [18]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

tokenizer = AutoTokenizer.from_pretrained(model_id)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    attn_implementation="sdpa",                   # Change to Flash Attention if GPU has support
    dtype=torch.float16,                          # Change to bfloat16 if GPU has support
    use_cache=True,                               # Whether to cache attention outputs to speed up inference
    device_map={"": 0},                           # Pin explicitly to the single visible GPU
    quantization_config=BitsAndBytesConfig(
        load_in_4bit=True,                        # Load the model in 4-bit precision to save memory
        bnb_4bit_compute_dtype=torch.float16,     # Data type used for internal computations in quantization
        bnb_4bit_use_double_quant=True,           # Use double quantization to improve accuracy
        bnb_4bit_quant_type="nf4"                 # Type of quantization. "nf4" is recommended for recent LLMs
    )
)


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [19]:
from peft import LoraConfig

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
)


In [20]:
from trl import SFTConfig

training_args = SFTConfig(
    # Output
    output_dir=output_dir,

    # Training
    num_train_epochs=2,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,

    learning_rate=2e-4,
    warmup_ratio=0.03,

    optim="paged_adamw_8bit",
    lr_scheduler_type="cosine",

    # Sequence
    max_length=768,
    packing=False,

    # Eval
    eval_strategy="epoch",

    # Logging
    logging_steps=10,
    save_strategy="epoch",
    report_to="none",                             # Set to "wandb" only if you have logged in with an API key

    # Memory optimization
    use_liger_kernel=False,
    activation_offloading=False,

    # Reproducibility
    seed=42,

    # We are NOT publishing during training
    push_to_hub=False,
)


warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [21]:
from trl import SFTTrainer

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    peft_config=peft_config,
    processing_class=tokenizer,
)


Tokenizing train dataset:   0%|          | 0/1800 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/1800 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/200 [00:00<?, ? examples/s]

Building labels for eval dataset:   0%|          | 0/200 [00:00<?, ? examples/s]

In [22]:
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)

print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")


GPU = Tesla T4. Max memory = 14.562 GB.
2.82 GB of memory reserved.


In [23]:
trainer_stats = trainer.train()


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Epoch,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
1,0.071443,0.072041,0.080669,493894.000000,0.980784
2,0.046008,0.045663,0.058587,987788.000000,0.986234


In [24]:
model.eval()
model.config.use_cache = True

# Token ids for a bare "A", "B", "C", "D", "E" as the model would emit them right after "Answer:"
# Some tokenizers prefix a leading space depending on context, so we check both forms and use
# whichever each one actually maps to as a single token.
def letter_token_id(letter):
    for candidate in (letter, " " + letter):
        ids = tokenizer.encode(candidate, add_special_tokens=False)
        if len(ids) == 1:
            return ids[0]
    # fallback: just take the last token of the multi-token encoding
    return tokenizer.encode(letter, add_special_tokens=False)[-1]

option_token_ids = {letter: letter_token_id(letter) for letter in OPTION_LETTERS}
print(option_token_ids)


{'A': 32, 'B': 33, 'C': 34, 'D': 35, 'E': 36}


In [25]:
# Left-padding is required for batched causal-LM scoring, since we need the LAST real
# (non-pad) token of every sequence in the batch to line up at the same index.
tokenizer.padding_side = "left"


In [45]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

tokenizer = AutoTokenizer.from_pretrained(model_id)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    attn_implementation="sdpa",
    torch_dtype=torch.float16,
    use_cache=True,
    device_map={"": 0},
    quantization_config=BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
    ),
)

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

In [47]:
from peft import PeftModel

model = PeftModel.from_pretrained(
    base_model,
    f"{output_dir}/checkpoint-450",
)

model.eval()

print("Loaded fine-tuned model.")
print(next(model.parameters()).device)

Loaded fine-tuned model.
cuda:0


In [49]:
from tqdm import tqdm
@torch.no_grad()
def predict_top3_batch(rows, batch_size=1):
    """rows: list of pandas Series (test rows). Returns list of ranked top-3 strings, one per row."""
    all_predictions = []

    for start in tqdm(range(0, len(rows), batch_size)):
        batch_rows = rows[start:start + batch_size]

        texts = [
            tokenizer.apply_chat_template(
                [{"role": "user", "content": build_prompt(row)}],
                add_generation_prompt=True,
                tokenize=False,
            )
            for row in batch_rows
        ]

        encoded = tokenizer(
            texts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=320,
        ).to(model.device)
        with torch.inference_mode():
            outputs = model(**encoded)
        # With left-padding, the last token of every sequence is at index -1 regardless of length
        last_token_logits = outputs.logits[:, -1, :]

        option_ids = torch.tensor([option_token_ids[letter] for letter in OPTION_LETTERS], device=model.device)
        option_logits = last_token_logits[:, option_ids]          # (batch, 5)
        probs = torch.softmax(option_logits, dim=-1)              # (batch, 5)

        ranked_idx = torch.argsort(probs, dim=-1, descending=True)[:, :3]   # (batch, 3)

        for row_idx in ranked_idx.tolist():
            top3 = [OPTION_LETTERS[i] for i in row_idx]
            all_predictions.append(" ".join(top3))

    return all_predictions


# Quick sanity check on a small slice before running over the full test set
sample_rows = [test.iloc[i] for i in range(4)]
print(predict_top3_batch(sample_rows, batch_size=4))


100%|██████████| 1/1 [00:00<00:00,  1.28it/s]

['A C D', 'B C D', 'C B A', 'E C D']


In [58]:
import re
import torch
from tqdm.auto import tqdm

OPTION_LETTERS = ["A", "B", "C", "D", "E"]

PREFIXES = [
    "Pick the best possible answer:",
    "Determine the correct option:",
    "Select the most accurate option:",
    "Identify the correct statement:",
]

def clean_prompt(prompt):
    prompt = prompt.strip()
    for p in PREFIXES:
        if prompt.startswith(p):
            return prompt[len(p):].strip()
    return prompt


@torch.inference_mode()
def predict_generate(rows):

    preds = []

    for row in tqdm(rows):

        question = clean_prompt(row["prompt"])

        prompt = f"""You are an expert at solving difficult multiple choice questions.

Read the question carefully.

Question:
{question}

Options:

A. {row["A"]}

B. {row["B"]}

C. {row["C"]}

D. {row["D"]}

E. {row["E"]}

Rank ALL FIVE options from BEST to WORST.

Output ONLY the option letters separated by spaces.

Example:
A C D B E

Do not explain.
"""

        text = tokenizer.apply_chat_template(
            [{"role": "user", "content": prompt}],
            tokenize=False,
            add_generation_prompt=True,
        )

        inputs = tokenizer(
            text,
            return_tensors="pt",
        ).to(model.device)

        outputs = model.generate(
            **inputs,
            max_new_tokens=10,
            do_sample=False,
            num_beams=5,
            num_return_sequences=3,
            early_stopping=True,
            pad_token_id=tokenizer.eos_token_id,
        )

        decoded = [
            tokenizer.decode(
                seq[inputs.input_ids.shape[1]:],
                skip_special_tokens=True,
            )
            for seq in outputs
        ]

        votes = {l: 0.0 for l in OPTION_LETTERS}

        for ans in decoded:

            letters = []

            for l in re.findall(r"[ABCDE]", ans.upper()):
                if l not in letters:
                    letters.append(l)

            if len(letters) > 0:
                votes[letters[0]] += 3

            if len(letters) > 1:
                votes[letters[1]] += 2

            if len(letters) > 2:
                votes[letters[2]] += 1

            if len(letters) > 3:
                votes[letters[3]] += 0.5

            if len(letters) > 4:
                votes[letters[4]] += 0.25

        ranking = sorted(
            OPTION_LETTERS,
            key=lambda x: votes[x],
            reverse=True,
        )

        preds.append(" ".join(ranking[:3]))

    return preds

In [59]:
predictions = predict_generate(test_rows)

submission = pd.DataFrame({
    "ID": test["id"],
    "Prediction": predictions
})

submission.to_csv("submission_beam.csv", index=False)

submission.head()

  0%|          | 0/500 [00:00<?, ?it/s]

,ID,Prediction
0,1,A E B
1,2,E B D
2,3,B C A
3,4,E C A
4,5,C D E


In [50]:
from tqdm.auto import tqdm

test_rows = [row for _, row in test.iterrows()]

# Tune batch_size to your GPU's VRAM headroom — 16 is a safe starting point for a 3B model
# on a T4 at max_length=768. Drop to 8 if you hit OOM, raise it if you have room to spare.


  0%|          | 0/500 [00:00<?, ?it/s]

,ID,Prediction
0,1,A E B
1,2,B E A
2,3,B A C
3,4,E C D
4,5,C D A
